# ONNX for Generative AI — Apply

This hands-on notebook demonstrates building and running a complete diffusion-style
image generation pipeline using ONNX Runtime. We implement text encoding, iterative
denoising, VAE decoding, and measure performance characteristics.

## 1. Setup

```
┌───────────────────────────────────────────────────────────────┐
│          GENERATIVE AI ONNX PIPELINE                           │
├───────────────────────────────────────────────────────────────┤
│                                                                 │
│  Components to build:                                           │
│  1. Text Encoder (CLIP-like) → text_encoder.onnx              │
│  2. UNet (noise predictor)   → unet.onnx                     │
│  3. VAE Decoder              → vae_decoder.onnx               │
│  4. Scheduler (DDIM)         → Python logic                   │
│  5. Pipeline orchestrator    → Python logic                   │
│                                                                 │
└───────────────────────────────────────────────────────────────┘
```

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort

os.makedirs('outputs', exist_ok=True)
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"ONNX Runtime: {ort.__version__}")
print(f"Providers: {ort.get_available_providers()}")

## 2. Text Encoder Component

The text encoder converts token IDs to dense embeddings that guide generation:

$$\text{TextEncoder}: \mathbb{Z}^{B \times L} \rightarrow \mathbb{R}^{B \times L \times D}$$

Where $L=77$ (max sequence length) and $D=768$ (embedding dimension).

In [ ]:
class TextEncoder(nn.Module):
    """CLIP-style text encoder for conditioning."""
    
    def __init__(self, vocab_size=49408, embed_dim=768, max_len=77, 
                 num_layers=4, num_heads=12):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(max_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 4, activation='gelu',
            batch_first=True, dropout=0.0
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.final_norm = nn.LayerNorm(embed_dim)
    
    def forward(self, input_ids):
        seq_len = input_ids.shape[1]
        pos_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        x = self.token_embedding(input_ids) + self.position_embedding(pos_ids)
        
        # Causal mask for CLIP-style encoding
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device), diagonal=1)
        
        x = self.encoder(x, mask=causal_mask)
        return self.final_norm(x)

text_encoder = TextEncoder(embed_dim=512, num_layers=3, num_heads=8)
text_encoder.eval()
print(f"Text Encoder: {sum(p.numel() for p in text_encoder.parameters()):,} params")

# Export
te_path = 'outputs/text_encoder.onnx'
dummy_tokens = torch.randint(0, 49408, (1, 77))
torch.onnx.export(
    text_encoder, dummy_tokens, te_path,
    input_names=['input_ids'], output_names=['text_embeddings'],
    dynamic_axes={'input_ids': {0:'batch'}, 'text_embeddings': {0:'batch'}},
    opset_version=14, do_constant_folding=True
)
print(f"✓ Exported: {os.path.getsize(te_path)/1024/1024:.2f} MB")

## 3. UNet Noise Predictor

The UNet takes a noisy latent, timestep, and text conditioning to predict noise:

$$\epsilon_\theta(x_t, t, c): \mathbb{R}^{B \times 4 \times H \times W} \times \mathbb{Z} \times \mathbb{R}^{B \times 77 \times D} \rightarrow \mathbb{R}^{B \times 4 \times H \times W}$$

```
Inputs:                      Output:
┌──────────────┐            ┌──────────────┐
│ latent       │            │ noise_pred   │
│ [1,4,32,32]  │──┐        │ [1,4,32,32]  │
└──────────────┘  │  UNet  └──────────────┘
┌──────────────┐  ├──────▶
│ timestep [1] │──┤
└──────────────┘  │
┌──────────────┐  │
│ context      │──┘
│ [1,77,512]   │
└──────────────┘
```

In [ ]:
class SinusoidalTimeEmbed(nn.Module):
    """Sinusoidal timestep embedding."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(nn.Linear(dim, dim*4), nn.SiLU(), nn.Linear(dim*4, dim))
    
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device).float() / half)
        emb = t.float()[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return self.proj(emb)

class ResBlock(nn.Module):
    def __init__(self, channels, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
    
    def forward(self, x, t_emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = F.silu(self.norm2(h))
        h = self.conv2(h)
        return h + x

class CrossAttnBlock(nn.Module):
    def __init__(self, channels, context_dim=512, heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.to_q = nn.Linear(channels, channels)
        self.to_k = nn.Linear(context_dim, channels)
        self.to_v = nn.Linear(context_dim, channels)
        self.to_out = nn.Linear(channels, channels)
        self.heads = heads
    
    def forward(self, x, context):
        B, C, H, W = x.shape
        h = self.norm(x)
        h = h.view(B, C, H*W).transpose(1, 2)  # [B, HW, C]
        
        q = self.to_q(h)
        k = self.to_k(context)
        v = self.to_v(context)
        
        # Multi-head attention
        head_dim = C // self.heads
        q = q.view(B, -1, self.heads, head_dim).transpose(1, 2)
        k = k.view(B, -1, self.heads, head_dim).transpose(1, 2)
        v = v.view(B, -1, self.heads, head_dim).transpose(1, 2)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        
        out = out.transpose(1, 2).reshape(B, H*W, C)
        out = self.to_out(out)
        out = out.transpose(1, 2).view(B, C, H, W)
        return x + out

class DiffusionUNet(nn.Module):
    """Simplified UNet for diffusion."""
    def __init__(self, in_ch=4, base_ch=128, time_dim=256, context_dim=512):
        super().__init__()
        self.time_embed = SinusoidalTimeEmbed(time_dim)
        
        # Input
        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        
        # Down
        self.down1 = ResBlock(base_ch, time_dim)
        self.attn1 = CrossAttnBlock(base_ch, context_dim)
        self.down_conv = nn.Conv2d(base_ch, base_ch*2, 3, stride=2, padding=1)
        
        # Mid
        self.mid1 = ResBlock(base_ch*2, time_dim)
        self.mid_attn = CrossAttnBlock(base_ch*2, context_dim)
        self.mid2 = ResBlock(base_ch*2, time_dim)
        
        # Up
        self.up_conv = nn.ConvTranspose2d(base_ch*2, base_ch, 2, stride=2)
        self.up1 = ResBlock(base_ch*2, time_dim)  # *2 for skip connection
        self.attn_up = CrossAttnBlock(base_ch*2, context_dim)
        
        # Output
        self.out_norm = nn.GroupNorm(8, base_ch*2)
        self.out_conv = nn.Conv2d(base_ch*2, in_ch, 3, padding=1)
    
    def forward(self, x, timestep, context):
        t_emb = self.time_embed(timestep)
        
        # Encode
        h = self.in_conv(x)
        h1 = self.down1(h, t_emb)
        h1 = self.attn1(h1, context)
        h = self.down_conv(h1)
        
        # Middle
        h = self.mid1(h, t_emb)
        h = self.mid_attn(h, context)
        h = self.mid2(h, t_emb)
        
        # Decode with skip
        h = self.up_conv(h)
        h = torch.cat([h, h1], dim=1)  # Skip connection
        h = self.up1(h, t_emb)
        h = self.attn_up(h, context)
        
        h = F.silu(self.out_norm(h))
        return self.out_conv(h)

unet = DiffusionUNet(in_ch=4, base_ch=64, time_dim=256, context_dim=512)
unet.eval()
print(f"UNet: {sum(p.numel() for p in unet.parameters()):,} params")

# Test
with torch.no_grad():
    test_out = unet(torch.randn(1,4,32,32), torch.tensor([500]), torch.randn(1,77,512))
print(f"UNet output: {test_out.shape}")

In [ ]:
# Export UNet
unet_path = 'outputs/unet.onnx'
torch.onnx.export(
    unet,
    (torch.randn(1,4,32,32), torch.tensor([500]), torch.randn(1,77,512)),
    unet_path,
    input_names=['latent', 'timestep', 'context'],
    output_names=['noise_pred'],
    dynamic_axes={
        'latent': {0: 'batch'},
        'context': {0: 'batch'},
        'noise_pred': {0: 'batch'}
    },
    opset_version=14,
    do_constant_folding=True
)
onnx.checker.check_model(onnx.load(unet_path))
print(f"✓ UNet exported: {os.path.getsize(unet_path)/1024/1024:.2f} MB")

## 4. VAE Decoder

Decodes latent representations back to pixel space:

$$\text{VAE Decoder}: \mathbb{R}^{B \times 4 \times H/8 \times W/8} \rightarrow \mathbb{R}^{B \times 3 \times H \times W}$$

Spatial upsampling factor: 8× (32×32 latent → 256×256 image)

In [ ]:
class VAEDecoder(nn.Module):
    """VAE Decoder: latent → image."""
    def __init__(self, latent_ch=4, base_ch=128, out_ch=3):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Conv2d(latent_ch, base_ch, 3, padding=1),
            nn.GroupNorm(8, base_ch),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),  # 2×
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.GroupNorm(8, base_ch),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),  # 4×
            nn.Conv2d(base_ch, base_ch//2, 3, padding=1),
            nn.GroupNorm(8, base_ch//2),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),  # 8×
            nn.Conv2d(base_ch//2, out_ch, 3, padding=1),
            nn.Tanh()  # Output in [-1, 1]
        )
    
    def forward(self, latent):
        return self.decoder(latent)

vae = VAEDecoder()
vae.eval()
print(f"VAE Decoder: {sum(p.numel() for p in vae.parameters()):,} params")

# Export
vae_path = 'outputs/vae_decoder.onnx'
torch.onnx.export(
    vae, torch.randn(1, 4, 32, 32), vae_path,
    input_names=['latent'], output_names=['image'],
    dynamic_axes={'latent': {0:'batch'}, 'image': {0:'batch'}},
    opset_version=14, do_constant_folding=True
)
print(f"✓ VAE Decoder exported: {os.path.getsize(vae_path)/1024/1024:.2f} MB")

## 5. DDIM Scheduler

The scheduler controls the denoising process:

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \cdot \hat{x}_0 + \sqrt{1 - \bar\alpha_{t-1}} \cdot \hat\epsilon$$

Where $\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\epsilon_\theta}{\sqrt{\bar\alpha_t}}$

In [ ]:
class DDIMScheduler:
    """DDIM scheduler for diffusion sampling."""
    
    def __init__(self, num_train_steps=1000, beta_start=0.00085, beta_end=0.012):
        # Linear schedule in sqrt space (SD-style)
        betas = np.linspace(beta_start**0.5, beta_end**0.5, num_train_steps)**2
        alphas = 1.0 - betas
        self.alphas_cumprod = np.cumprod(alphas).astype(np.float32)
        self.num_train_steps = num_train_steps
    
    def set_timesteps(self, num_inference_steps):
        """Create evenly-spaced timestep schedule."""
        step_ratio = self.num_train_steps // num_inference_steps
        self.timesteps = np.arange(0, num_inference_steps)[::-1] * step_ratio
        self.timesteps = self.timesteps.astype(np.int64)
    
    def step(self, noise_pred, timestep, sample, eta=0.0):
        """Perform one DDIM step."""
        t = timestep
        prev_t = t - self.num_train_steps // len(self.timesteps)
        
        alpha_prod_t = self.alphas_cumprod[t]
        alpha_prod_t_prev = self.alphas_cumprod[max(prev_t, 0)] if prev_t >= 0 else 1.0
        
        # Predict x_0
        pred_x0 = (sample - np.sqrt(1 - alpha_prod_t) * noise_pred) / np.sqrt(alpha_prod_t)
        
        # Clip for stability
        pred_x0 = np.clip(pred_x0, -5.0, 5.0)
        
        # Direction pointing to x_t
        pred_dir = np.sqrt(1 - alpha_prod_t_prev) * noise_pred
        
        # x_{t-1}
        prev_sample = np.sqrt(alpha_prod_t_prev) * pred_x0 + pred_dir
        
        return prev_sample

scheduler = DDIMScheduler()
scheduler.set_timesteps(20)
print(f"DDIM Schedule (20 steps):")
print(f"  Timesteps: {scheduler.timesteps[:5]}...{scheduler.timesteps[-5:]}")
print(f"  Alpha_bar range: [{scheduler.alphas_cumprod[scheduler.timesteps[-1]]:.4f}, "
      f"{scheduler.alphas_cumprod[scheduler.timesteps[0]]:.4f}]")

## 6. Complete ONNX Diffusion Pipeline

```
┌─────────────────────────────────────────────────────────────┐
│              DIFFUSION GENERATION PIPELINE                    │
├─────────────────────────────────────────────────────────────┤
│                                                               │
│  1. Tokenize prompt → token_ids [1, 77]                     │
│  2. text_encoder.onnx(token_ids) → embeddings [1, 77, 512]  │
│  3. Initialize z_T ~ N(0, I) [1, 4, 32, 32]                │
│  4. For each timestep t in schedule:                        │
│     a. unet.onnx(z_t, t, embeddings) → ε_cond             │
│     b. unet.onnx(z_t, t, zeros)      → ε_uncond           │
│     c. ε = ε_uncond + w*(ε_cond - ε_uncond)  [CFG]        │
│     d. z_{t-1} = scheduler.step(ε, t, z_t)                │
│  5. vae_decoder.onnx(z_0) → image [1, 3, 256, 256]        │
│  6. Post-process: (image + 1) / 2 * 255 → uint8           │
│                                                               │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
class ONNXDiffusionPipeline:
    """Complete diffusion pipeline using ONNX Runtime."""
    
    def __init__(self, text_encoder_path, unet_path, vae_path,
                 num_steps=20, guidance_scale=7.5):
        self.text_session = ort.InferenceSession(text_encoder_path)
        self.unet_session = ort.InferenceSession(unet_path)
        self.vae_session = ort.InferenceSession(vae_path)
        
        self.scheduler = DDIMScheduler()
        self.scheduler.set_timesteps(num_steps)
        self.guidance_scale = guidance_scale
        self.num_steps = num_steps
    
    def encode_text(self, token_ids):
        """Encode text tokens."""
        return self.text_session.run(None, {'input_ids': token_ids})[0]
    
    def predict_noise(self, latent, timestep, embeddings):
        """UNet forward pass."""
        return self.unet_session.run(None, {
            'latent': latent.astype(np.float32),
            'timestep': np.array([timestep], dtype=np.int64),
            'context': embeddings.astype(np.float32)
        })[0]
    
    def decode_latent(self, latent):
        """VAE decode."""
        return self.vae_session.run(None, {'latent': latent.astype(np.float32)})[0]
    
    def generate(self, token_ids, latent_shape=(1, 4, 32, 32), seed=42, callback=None):
        """Full generation pipeline."""
        np.random.seed(seed)
        timings = {'text_encode': 0, 'unet': 0, 'scheduler': 0, 'vae': 0}
        
        # Step 1: Encode text
        t0 = time.perf_counter()
        text_emb = self.encode_text(token_ids)
        uncond_emb = np.zeros_like(text_emb)
        timings['text_encode'] = (time.perf_counter() - t0) * 1000
        
        # Step 2: Initialize noise
        latent = np.random.randn(*latent_shape).astype(np.float32)
        
        # Step 3: Iterative denoising
        for i, t in enumerate(self.scheduler.timesteps):
            # Conditional prediction
            t0 = time.perf_counter()
            noise_cond = self.predict_noise(latent, t, text_emb)
            noise_uncond = self.predict_noise(latent, t, uncond_emb)
            timings['unet'] += (time.perf_counter() - t0) * 1000
            
            # Classifier-free guidance
            noise_pred = noise_uncond + self.guidance_scale * (noise_cond - noise_uncond)
            
            # Scheduler step
            t0 = time.perf_counter()
            latent = self.scheduler.step(noise_pred, int(t), latent)
            timings['scheduler'] += (time.perf_counter() - t0) * 1000
            
            if callback:
                callback(i, self.num_steps, latent)
        
        # Step 4: VAE decode
        t0 = time.perf_counter()
        image = self.decode_latent(latent)
        timings['vae'] = (time.perf_counter() - t0) * 1000
        
        # Step 5: Post-process
        image = ((image + 1.0) / 2.0 * 255).clip(0, 255).astype(np.uint8)
        
        return image, timings

# Create pipeline
pipeline = ONNXDiffusionPipeline(
    te_path, unet_path, vae_path,
    num_steps=10, guidance_scale=7.5
)
print("✓ Pipeline created with all ONNX components")

In [ ]:
# Run generation
def progress_callback(step, total, latent):
    if step % 3 == 0:
        print(f"  Step {step+1}/{total}: latent std={np.std(latent):.4f}")

fake_tokens = np.random.randint(0, 49408, (1, 77)).astype(np.int64)

print("Generating image...")
image, timings = pipeline.generate(fake_tokens, seed=42, callback=progress_callback)

print(f"\n✓ Generation complete!")
print(f"  Output shape: {image.shape} (B, C, H, W)")
print(f"  Value range: [{image.min()}, {image.max()}]")
print(f"\nTiming Breakdown:")
total = sum(timings.values())
for component, ms in timings.items():
    pct = ms / total * 100
    bar = '█' * int(pct / 3)
    print(f"  {component:>14}: {ms:>8.1f} ms ({pct:>5.1f}%) {bar}")
print(f"  {'TOTAL':>14}: {total:>8.1f} ms")

## 7. Benchmarking Inference Steps

Compare generation quality/speed trade-offs:

$$\text{Total Time} \approx T_{\text{text}} + 2N \cdot T_{\text{unet}} + T_{\text{vae}}$$

Where $N$ = number of steps, factor 2 for CFG.

In [ ]:
# Benchmark different step counts
print("Step Count vs Generation Time")
print("=" * 55)
print(f"{'Steps':>6} {'Total (ms)':>12} {'UNet %':>8} {'ms/step':>10} {'Est. 50 steps':>14}")
print("-" * 55)

for num_steps in [4, 8, 10, 15, 20]:
    pipe = ONNXDiffusionPipeline(te_path, unet_path, vae_path,
                                  num_steps=num_steps, guidance_scale=7.5)
    _, timings = pipe.generate(fake_tokens, seed=42)
    total = sum(timings.values())
    unet_pct = timings['unet'] / total * 100
    ms_per_step = timings['unet'] / num_steps
    est_50 = timings['text_encode'] + ms_per_step * 50 + timings['vae']
    print(f"{num_steps:>6} {total:>12.1f} {unet_pct:>7.1f}% {ms_per_step:>10.1f} {est_50:>12.0f}ms")

## 8. Quantization for Speed

Apply INT8 quantization to reduce memory and increase speed:

$$W_{int8} = \text{round}(W_{fp32} / s) + z$$

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

# Quantize UNet (the bottleneck)
unet_int8_path = 'outputs/unet_int8.onnx'
quantize_dynamic(unet_path, unet_int8_path, weight_type=QuantType.QInt8)

# Compare sizes
fp32_size = os.path.getsize(unet_path) / 1024 / 1024
int8_size = os.path.getsize(unet_int8_path) / 1024 / 1024
print(f"UNet Quantization:")
print(f"  FP32: {fp32_size:.2f} MB")
print(f"  INT8: {int8_size:.2f} MB")
print(f"  Ratio: {fp32_size/int8_size:.2f}×")

# Benchmark INT8 UNet
unet_int8_session = ort.InferenceSession(unet_int8_path)
test_data = {
    'latent': np.random.randn(1,4,32,32).astype(np.float32),
    'timestep': np.array([500], dtype=np.int64),
    'context': np.random.randn(1,77,512).astype(np.float32)
}

# Warmup
for _ in range(5):
    unet_int8_session.run(None, test_data)

fp32_times = []
int8_times = []
unet_fp32_session = ort.InferenceSession(unet_path)
for _ in range(20):
    t0 = time.perf_counter()
    unet_fp32_session.run(None, test_data)
    fp32_times.append((time.perf_counter()-t0)*1000)
    
    t0 = time.perf_counter()
    unet_int8_session.run(None, test_data)
    int8_times.append((time.perf_counter()-t0)*1000)

print(f"\nSpeed comparison (single UNet call):")
print(f"  FP32: {np.mean(fp32_times):.2f} ms")
print(f"  INT8: {np.mean(int8_times):.2f} ms")
print(f"  Speedup: {np.mean(fp32_times)/np.mean(int8_times):.2f}×")

## 9. Image Variations (Img2Img)

Start from a partially noised image instead of pure noise:

$$x_t = \sqrt{\bar\alpha_t} x_0 + \sqrt{1-\bar\alpha_t} \epsilon$$

The `strength` parameter controls how much of the original is preserved:
- strength=1.0 → pure noise (text-to-image)
- strength=0.5 → 50% original structure kept
- strength=0.2 → 80% original preserved (subtle changes)

In [ ]:
def img2img_generate(pipeline, token_ids, init_latent, strength=0.75, seed=42):
    """Image-to-image generation."""
    np.random.seed(seed)
    
    # Determine starting timestep based on strength
    num_steps = pipeline.num_steps
    start_step = int(num_steps * (1 - strength))
    timesteps = pipeline.scheduler.timesteps[start_step:]
    
    # Add noise to init latent at the starting timestep
    t_start = timesteps[0]
    alpha_bar = pipeline.scheduler.alphas_cumprod[t_start]
    noise = np.random.randn(*init_latent.shape).astype(np.float32)
    latent = np.sqrt(alpha_bar) * init_latent + np.sqrt(1 - alpha_bar) * noise
    
    # Encode text
    text_emb = pipeline.encode_text(token_ids)
    uncond_emb = np.zeros_like(text_emb)
    
    # Denoise
    for t in timesteps:
        noise_cond = pipeline.predict_noise(latent, t, text_emb)
        noise_uncond = pipeline.predict_noise(latent, t, uncond_emb)
        noise_pred = noise_uncond + pipeline.guidance_scale * (noise_cond - noise_uncond)
        latent = pipeline.scheduler.step(noise_pred, int(t), latent)
    
    # Decode
    image = pipeline.decode_latent(latent)
    return ((image + 1) / 2 * 255).clip(0, 255).astype(np.uint8)

# Test img2img
init_latent = np.random.randn(1, 4, 32, 32).astype(np.float32)

print("Image-to-Image Generation:")
print("=" * 50)
for strength in [0.3, 0.5, 0.75, 1.0]:
    start = time.perf_counter()
    result = img2img_generate(pipeline, fake_tokens, init_latent, strength=strength)
    elapsed = (time.perf_counter() - start) * 1000
    steps_run = int(pipeline.num_steps * strength)
    print(f"  strength={strength:.2f}: {steps_run} steps, {elapsed:.0f}ms, output={result.shape}")

## 10. Batch Generation

Generate multiple images in parallel for throughput:

$$\text{Throughput} = \frac{B}{T(B)} > \frac{1}{T(1)}$$

In [ ]:
# Test batch generation
print("Batch Generation Performance")
print("=" * 55)
print(f"{'Batch':>6} {'Total (ms)':>12} {'Per Image':>12} {'Throughput':>14}")
print("-" * 55)

for batch_size in [1, 2, 4]:
    tokens = np.random.randint(0, 49408, (batch_size, 77)).astype(np.int64)
    latent_shape = (batch_size, 4, 32, 32)
    
    start = time.perf_counter()
    
    # Text encode
    text_emb = pipeline.text_session.run(None, {'input_ids': tokens})[0]
    uncond_emb = np.zeros_like(text_emb)
    
    # Initialize
    latent = np.random.randn(*latent_shape).astype(np.float32)
    
    # Denoise (5 steps for benchmark)
    scheduler = DDIMScheduler()
    scheduler.set_timesteps(5)
    for t in scheduler.timesteps:
        n_c = pipeline.unet_session.run(None, {
            'latent': latent, 'timestep': np.array([t], dtype=np.int64), 'context': text_emb})[0]
        n_u = pipeline.unet_session.run(None, {
            'latent': latent, 'timestep': np.array([t], dtype=np.int64), 'context': uncond_emb})[0]
        noise = n_u + 7.5 * (n_c - n_u)
        latent = scheduler.step(noise, int(t), latent)
    
    # Decode
    image = pipeline.vae_session.run(None, {'latent': latent.astype(np.float32)})[0]
    
    elapsed = (time.perf_counter() - start) * 1000
    per_image = elapsed / batch_size
    throughput = batch_size / (elapsed / 1000)
    print(f"{batch_size:>6} {elapsed:>12.1f} {per_image:>12.1f} {throughput:>12.2f}/s")

## 11. Model Profiling

Identify which operations consume the most time.

In [ ]:
# Analyze model structures
def analyze_onnx(path):
    m = onnx.load(path)
    ops = {}
    for n in m.graph.node:
        ops[n.op_type] = ops.get(n.op_type, 0) + 1
    return len(m.graph.node), ops

print("ONNX Model Analysis")
print("=" * 60)
for name, path in [('Text Encoder', te_path), ('UNet', unet_path), ('VAE Decoder', vae_path)]:
    nodes, ops = analyze_onnx(path)
    size = os.path.getsize(path) / 1024 / 1024
    print(f"\n  {name} ({size:.2f} MB, {nodes} nodes):")
    for op, count in sorted(ops.items(), key=lambda x:-x[1])[:8]:
        print(f"    {op:<20}: {count}")

## 12. Seed Variation and Reproducibility

With DDIM (eta=0), generation is deterministic given the same seed:

$$\text{Same seed} + \text{Same prompt} + \text{Same scheduler} = \text{Same image}$$

In [ ]:
# Verify reproducibility
img1, _ = pipeline.generate(fake_tokens, seed=42)
img2, _ = pipeline.generate(fake_tokens, seed=42)
img3, _ = pipeline.generate(fake_tokens, seed=123)

print("Reproducibility Test:")
print(f"  Same seed (42 vs 42): max diff = {np.max(np.abs(img1.astype(int) - img2.astype(int)))}")
print(f"  Diff seed (42 vs 123): max diff = {np.max(np.abs(img1.astype(int) - img3.astype(int)))}")
print(f"  Identical with same seed: {np.array_equal(img1, img2)}")

## 13. Guidance Scale Effects

$$\tilde\epsilon = \epsilon_{uncond} + w \cdot (\epsilon_{cond} - \epsilon_{uncond})$$

| Scale | Effect |
|-------|--------|
| 1.0 | No guidance (unconditional) |
| 3.0 | Mild guidance |
| 7.5 | Standard (SD default) |
| 15.0 | Strong (may oversaturate) |
| 30.0+ | Very strong (artifacts) |

In [ ]:
# Compare guidance scales
print("Guidance Scale Comparison")
print("=" * 50)
print(f"{'Scale':>8} {'Mean pixel':>12} {'Std pixel':>12} {'Saturation':>12}")
print("-" * 50)

for scale in [1.0, 3.0, 7.5, 12.0, 20.0]:
    pipe = ONNXDiffusionPipeline(te_path, unet_path, vae_path,
                                  num_steps=5, guidance_scale=scale)
    img, _ = pipe.generate(fake_tokens, seed=42)
    mean_val = np.mean(img)
    std_val = np.std(img)
    saturation = np.mean((img == 0) | (img == 255)) * 100
    print(f"{scale:>8.1f} {mean_val:>12.1f} {std_val:>12.1f} {saturation:>10.1f}%")

## 14. Session Configuration for Optimal Performance

```
CPU Optimization:              GPU Optimization:
┌─────────────────────┐       ┌─────────────────────┐
│ intra_threads = N   │       │ CUDAExecutionProvider│
│ graph_opt = ALL     │       │ IO Binding           │
│ INT8 quantization   │       │ FP16 precision       │
│ Memory pattern opt  │       │ TensorRT subgraph    │
└─────────────────────┘       └─────────────────────┘
```

In [ ]:
import multiprocessing

# Test different thread configs for UNet
print(f"UNet Threading Benchmark (CPU cores: {multiprocessing.cpu_count()})")
print("=" * 50)

for threads in [1, 2, 4, multiprocessing.cpu_count()]:
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = threads
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    sess = ort.InferenceSession(unet_path, opts)
    
    # Warmup
    for _ in range(3):
        sess.run(None, test_data)
    
    times = []
    for _ in range(10):
        t0 = time.perf_counter()
        sess.run(None, test_data)
        times.append((time.perf_counter()-t0)*1000)
    
    print(f"  {threads} threads: {np.mean(times):.2f} ms (±{np.std(times):.2f})")

## 15. Autoregressive LLM Generation

Demonstrate token-by-token generation with ONNX.

In [ ]:
class SimpleLLM(nn.Module):
    def __init__(self, vocab_size=32000, dim=256, heads=8, layers=3, max_len=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.pos = nn.Embedding(max_len, dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim*4,
            activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)
    
    def forward(self, input_ids):
        B, S = input_ids.shape
        pos = torch.arange(S, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(S, device=x.device)
        x = self.transformer(x, mask=mask)
        x = self.norm(x)
        return self.head(x)

llm = SimpleLLM(vocab_size=32000)
llm.eval()

# Export
llm_path = 'outputs/llm.onnx'
torch.onnx.export(
    llm, torch.randint(0,32000,(1,16)), llm_path,
    input_names=['input_ids'], output_names=['logits'],
    dynamic_axes={'input_ids':{0:'b',1:'s'}, 'logits':{0:'b',1:'s'}},
    opset_version=14
)

# Generate
llm_session = ort.InferenceSession(llm_path)
prompt = np.array([[1, 100, 200, 300]], dtype=np.int64)
generated = list(prompt[0])

start = time.perf_counter()
for _ in range(30):
    logits = llm_session.run(None, {'input_ids': np.array([generated], dtype=np.int64)})[0]
    next_token = np.argmax(logits[0, -1, :])
    generated.append(int(next_token))
elapsed = (time.perf_counter() - start) * 1000

print(f"\nLLM Generation (no KV-cache):")
print(f"  Prompt: {len(prompt[0])} tokens")
print(f"  Generated: 30 new tokens")
print(f"  Time: {elapsed:.1f} ms")
print(f"  Speed: {30/(elapsed/1000):.1f} tokens/sec")
print(f"  Note: Without KV-cache, latency grows quadratically")

## 16. Summary and Production Deployment

```
┌──────────────────────────────────────────────────────────────┐
│        GENERATIVE AI DEPLOYMENT CHECKLIST                     │
├──────────────────────────────────────────────────────────────┤
│                                                                │
│  □ Export each component separately                           │
│  □ Validate outputs match PyTorch                            │
│  □ Apply FP16 for GPU / INT8 for CPU                        │
│  □ Profile: UNet dominates (80%+ of time)                    │
│  □ Use DDIM/DPM++ with 20-50 steps                          │
│  □ Implement CFG efficiently (batch=2)                       │
│  □ Add safety checks (NSFW filter)                           │
│  □ Set up async queue for GPU sharing                        │
│  □ Monitor GPU memory (especially KV-cache for LLMs)         │
│  □ Consider LCM/Turbo for real-time applications            │
│                                                                │
└──────────────────────────────────────────────────────────────┘
```

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("  GENERATIVE AI + ONNX APPLY - COMPLETE")
print("=" * 60)

print("\n  Exported Models:")
total_size = 0
for f in sorted(os.listdir('outputs')):
    if f.endswith('.onnx'):
        size = os.path.getsize(f'outputs/{f}') / 1024 / 1024
        total_size += size
        print(f"    {f:<30} {size:.2f} MB")
print(f"    {'TOTAL':<30} {total_size:.2f} MB")

print(f"""
  Key Results:
  • Complete diffusion pipeline running with ONNX Runtime
  • Text → Latent → Image generation working end-to-end
  • INT8 quantization reduces UNet size and improves speed
  • DDIM enables flexible step count (4-50 steps)
  • Batch processing improves throughput
  • Deterministic generation with fixed seeds
""")
print("✓ All generative AI ONNX workflows demonstrated!")